In [1]:
# Cell 1: Chọn device phù hợp (NVIDIA / AMD / CPU)

import math
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import random
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from matrix_correla import (
    load_traffic_tensor,
    compute_anchor_correlations,
    build_influenced_subgraph,
    build_laplacian_from_distance,
    build_zone
)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA-compatible GPU (NVIDIA hoặc AMD qua ROCm):", torch.cuda.get_device_name(0))
else:
    # Thử DirectML (Windows + AMD)
    try:
        import torch_directml
        device = torch_directml.device()
        print("Using DirectML device (có thể là AMD Radeon qua DirectML).")
    except ImportError:
        device = torch.device("cpu")
        print("Không tìm thấy GPU, dùng CPU.")

device


Using DirectML device (có thể là AMD Radeon qua DirectML).


device(type='privateuseone', index=0)

In [2]:
class TemporalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.d_model = d_model
        self.rel_pos_emb = nn.Embedding(max_len, d_model)
        self.time_proj = nn.Linear(4,d_model)
    def forward(self,time_of_day,day_of_week):
        B,L = time_of_day.shape

        # --- relative position embedding ---
        rel_pos = torch.arange(L, device=time_of_day.device) #(L,)
        rel_pos = rel_pos.unsqueeze(0).expand(B,L) # biến thành (1,L) rồi nhân bản thành (B,L)
        rel_pos_pe = self.rel_pos_emb(rel_pos)

        # --- sin/cos time-of-day (0..1) -> (0..2π) ---
        tod_angle = time_of_day * 2 *math.pi
        tod_sin = torch.sin(tod_angle)
        tod_cos = torch.cos(tod_angle)

        # --- sin/cos day-of-week (0..6) ---
        dow = day_of_week.float() / 5.0
        dow_angle = dow * 2 * math.pi
        dow_sin = torch.sin(dow_angle)
        dow_cos = torch.cos(dow_angle)
        t_feat = torch.stack([tod_sin, tod_cos, dow_sin, dow_cos], dim=-1)
        # project lên d_model
        t_pe = self.time_proj(t_feat)
        return rel_pos_pe + t_pe

In [3]:
class SpatialEncoding(nn.Module):
    """
    Spatial PE = Laplacian eigenvectors của subgraph.
    eigvec: (N, d_spa)
    """
    def __init__(self,d_model,d_spa):
        super().__init__()
        self.proj = nn.Linear(d_spa, d_model)
    def forward(self, lap_eigvec):
        """
        lap_eigvec: (N, d_spa)
        return: (N, d_model)
        """
        return self.proj(lap_eigvec)  # (N, d_model)


In [4]:
class SpatioTemporalEncoder(nn.Module):
    """
    Encoder-only Transformer với token = (segment, time).

    Input x: (B, L, N, d_in)
    Output H: (B, L, N, d_model)
    """
    def __init__(self, d_in, d_model, nhead,num_layers,dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.input_proj = nn.Linear(d_in, d_model)
        self.temp_enc = TemporalEncoding(d_model=d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,dim_feedforward=dim_feedforward,dropout=dropout)
        self.encoder = nn.TransformerEncoder(encoder_layer,num_layers=num_layers)

    def forward(self,x,time_of_day,day_of_week,spatial_pe):
        """
        x: (B, L, N, d_in)
        time_of_day: (B, L)
        day_of_week: (B, L)
        spatial_pe: (N, d_model) – output từ SpatialEncoding

        return: H (B, L, N, d_model)
        """
        B,L,N,d_in = x.shape
        x_proj = self.input_proj(x) # (B, L, N, d_model)

        # 2) Temporal PE
        t_pe = self.temp_enc(time_of_day,day_of_week)
        t_pe = t_pe.unsqueeze(2) # (B, L, 1, self.d_model)

        # 3) Spatial PE
        s_pe = spatial_pe.unsqueeze(0).unsqueeze(0)             # spatial_pe: (N, d_model)
                                                                # s_pe:       (1, 1, N, d_model)

        # 4) Sum PE + feature
        h = x_proj + t_pe + s_pe

        # 5) Flatten (L, N) -> S = L*N
        h = h.view(B, L * N, self.d_model)

        # 6) Transformer encoder
        H = self.encoder(h)                       # (B, S, d_model)

        # 7) Reshape lại (B, L, N, d_model)
        H = H.view(B,L,N,self.d_model)
        return H




In [5]:
# Cell 5: Prediction Head + Full SpatioTemporalTransformer

class PredictionHead(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, H_last):      # H_last: (B, N, d_model)
        B, N, D = H_last.shape
        logits = self.mlp(H_last.reshape(B*N, D))  # (B*N,1)
        return logits.reshape(B, N)                # (B,N)


class SpatioTemporalTransformer(nn.Module):
    """
    Full model:
      - Encoder-only Transformer
      - Spatial PE = Laplacian eigenvectors
      - Temporal PE = relative + time-of-day + day-of-week
      - Dự đoán congestion tương lai cho anchor (node index anchor_idx)
    """
    def __init__(
        self,
        d_in,
        d_model=128,
        d_spa=16,
        nhead=4,
        num_layers=3,
        dim_feedforward=512,
        dropout=0.1,
        hidden_dim_head=128,
        anchor_idx=0,
    ):
        super().__init__()
        self.anchor_idx = anchor_idx

        self.spatial_enc = SpatialEncoding(d_model=d_model, d_spa=d_spa)

        self.encoder = SpatioTemporalEncoder(
            d_in=d_in,
            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
        )

        self.head = PredictionHead(d_model=d_model, hidden_dim=hidden_dim_head)

    def forward(self, x, time_of_day, day_of_week, lap_eigvec):
        spatial_pe = self.spatial_enc(lap_eigvec)
        H = self.encoder(x, time_of_day, day_of_week, spatial_pe)
        H_last = H[:, -1, :, :]     # (B,N,d_model)
        logits = self.head(H_last)  # (B,N)
        return logits


In [6]:
BASE_DIR = Path.cwd().parents[2]
OUTPUT_DIR = BASE_DIR / "data" / "processed" / "tomtom_stats"


SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)


In [7]:
class ZoneDataset(Dataset):
    def __init__(
        self,
        output_dir,
        split_indices,
        L=4,
        delta=1,
        tau_max=10,
        Wmin=0.5,
        top_k=60,
    ):
        self.output_dir = Path(output_dir)
        data = load_traffic_tensor(self.output_dir)

        self.values = data["values"]        # (T,N)
        self.segment_ids = data["segment_ids"]
        self.T = self.values.shape[0]

        meta = np.load(self.output_dir / "traffic_tensor.npz")
        self.time_of_day = meta["time_of_day"]
        self.day_of_week = meta["day_of_week"]
        self.is_congested = meta["is_congested"]

        self.L = L
        self.delta = delta
        self.split_indices = split_indices

        self.tau_max = tau_max
        self.Wmin = Wmin
        self.top_k = top_k

    def __len__(self):
        return len(self.split_indices)

    def __getitem__(self, idx):
        t = self.split_indices[idx]

        # ----------- Build zone ----------
        seed_seg_id = int(np.random.choice(self.segment_ids))
        zone = build_zone(
            output_dir=self.output_dir,
            seed_seg_id=seed_seg_id,
            tau_max=self.tau_max,
            Wmin=self.Wmin,
            top_k=self.top_k,
            d_spa=16,
        )

        zone_idx = zone["zone_indices"]        # (N_zone,)
        lap_eigvec = zone["lap_eigvec"]        # (N_zone, d_spa)

        # ------------- Build time window --------------
        x = self.values[t-self.L+1 : t+1, zone_idx]  # (L, N_zone)
        tod = self.time_of_day[t-self.L+1 : t+1]     # (L,)
        dow = self.day_of_week[t-self.L+1 : t+1]     # (L,)

        # (optionally expand dims for d_in = 1)
        x = x[..., None]  # (L,N_zone,1)

        # label y = congestion at (t+delta)
        y = self.is_congested[t + self.delta, zone_idx].astype(np.float32)

        return {
            "x": torch.tensor(x, dtype=torch.float32),
            "tod": torch.tensor(tod, dtype=torch.float32),
            "dow": torch.tensor(dow, dtype=torch.float32),
            "lap": torch.tensor(lap_eigvec, dtype=torch.float32),
            "y": torch.tensor(y, dtype=torch.float32),
        }


In [8]:
# Cell: Collate function để pad theo N_zone lớn nhất trong batch
def collate_zone(batch):
    """
    batch = list of dict { x:(L,N_zone,1), tod:(L,), dow:(L,), lap:(N_zone,d_spa), y:(N_zone,) }
    Trả về tensors padded theo N_zone_max
    """
    L = batch[0]["x"].shape[0]
    d_in = batch[0]["x"].shape[-1]
    d_spa = batch[0]["lap"].shape[-1]

    # ---- tìm max N_zone trong batch
    Nmax = max(item["x"].shape[1] for item in batch)
    B = len(batch)

    # ---- init tensor padded
    x_pad  = torch.zeros(B, L, Nmax, d_in)
    lap_pad= torch.zeros(B, Nmax, d_spa)
    y_pad  = torch.full((B, Nmax), fill_value=-1.0)  # dùng -1 để mask label
    mask   = torch.zeros(B, Nmax)                    # 1=valid, 0=padded

    tod = torch.stack([item["tod"] for item in batch], dim=0)  # (B,L)
    dow = torch.stack([item["dow"] for item in batch], dim=0)  # (B,L)

    # ---- copy từng sample vào padded tensor
    for i, item in enumerate(batch):
        Ni = item["x"].shape[1]
        x_pad[i, :, :Ni, :] = item["x"]
        lap_pad[i, :Ni, :] = item["lap"]
        y_pad[i, :Ni] = item["y"]
        mask[i, :Ni] = 1.0

    return {
        "x": x_pad,
        "tod": tod,
        "dow": dow,
        "lap": lap_pad,
        "y": y_pad,
        "mask": mask
    }


In [9]:
def masked_bce_loss(logits, labels, mask):
    """
    logits: (B,N)
    labels: (B,N)
    mask:   (B,N)  — 1=valid, 0=padding
    """
    loss = F.binary_cross_entropy_with_logits(logits, labels, reduction="none")
    loss = loss * mask
    return loss.sum() / (mask.sum() + 1e-6)

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0

    for batch in loader:
        x   = batch["x"].to(device)
        tod = batch["tod"].to(device)
        dow = batch["dow"].to(device)
        lap = batch["lap"].to(device)
        y   = batch["y"].to(device)
        m   = batch["mask"].to(device)

        optimizer.zero_grad()

        logits = model(x, tod, dow, lap)  # (B,N)

        loss = masked_bce_loss(logits, y, m)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    total_loss = 0

    for batch in loader:
        x   = batch["x"].to(device)
        tod = batch["tod"].to(device)
        dow = batch["dow"].to(device)
        lap = batch["lap"].to(device)
        y   = batch["y"].to(device)
        m   = batch["mask"].to(device)

        logits = model(x, tod, dow, lap)
        loss = masked_bce_loss(logits, y, m)

        total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
import time
def build_splits(output_dir, L=3, delta=1):
    data = np.load(Path(output_dir) / "traffic_tensor.npz")
    T = data["values"].shape[0]

    # -------- ratio 80 / 10 / 10 ----------
    train_end = int(T * 0.80)
    valid_end = int(T * 0.90)

    def valid_indices(start, end):
        idxs = []
        for t in range(start, end):
            if (t - L + 1) < 0:
                continue
            if (t + delta) >= end:
                continue
            idxs.append(t)
        return idxs

    split_train = valid_indices(0, train_end)
    split_valid = valid_indices(train_end, valid_end)
    split_test  = valid_indices(valid_end, T)

    print("Total T =", T)
    print(f"Train={len(split_train)}, Valid={len(split_valid)}, Test={len(split_test)}")

    return split_train, split_valid, split_test
split_train, split_valid, split_test = build_splits(OUTPUT_DIR, L=3, delta=1)
train_ds = ZoneDataset(OUTPUT_DIR, split_train, L=3, delta=1)
valid_ds = ZoneDataset(OUTPUT_DIR, split_valid, L=3, delta=1)
test_ds  = ZoneDataset(OUTPUT_DIR, split_test,  L=3, delta=1)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  collate_fn=collate_zone)
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False, collate_fn=collate_zone)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False, collate_fn=collate_zone)
model = SpatioTemporalTransformer(
    d_in=1, d_model=128, d_spa=16, nhead=4, num_layers=3
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
EPOCHS = 20
for epoch in range(EPOCHS):
    t0 = time.time()
    loss_tr = train_one_epoch(model, train_loader, optimizer, device)
    loss_va = eval_epoch(model, valid_loader, device)
    t1 = time.time()
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss={loss_tr:.4f} | Valid Loss={loss_va:.4f}")


Total T = 528
Train=419, Valid=52, Test=52


C:\Users\ADMIN\miniconda3\envs\Specialized_Project_torchdml\lib\site-packages\torch\nn\modules\transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [ ]:
loss_test = eval_epoch(model, test_loader, device)
print(f"\n>>> FINAL TEST LOSS = {loss_test:.4f}")
